# Week 7: Hyperparameter Tuning
**Author:** Vishnu  
**Goal:** Optimise the stacking ensemble via grid search.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
project_path = '/content/drive/MyDrive/ParkinsonsProject/ParkinsonsDetection'
os.chdir(project_path)

import numpy as np
import joblib
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

Mounted at /content/drive


## Load Data

In [2]:
X_train = np.load('data/processed/X_train_fs.npy')
y_train = np.load('data/processed/y_train.npy')
X_test = np.load('data/processed/X_test_fs.npy')
y_test = np.load('data/processed/y_test.npy')

## Define Base Learners (same as before)

In [3]:
base_learners = [
    ('rf', RandomForestClassifier(random_state=42)),
    ('svm', SVC(probability=True, random_state=42)),
    ('knn', KNeighborsClassifier())
]

## Set Up Parameter Grid
We'll tune key hyperparameters for each base learner and the meta‑learner.

In [4]:
param_grid = {
    'rf__n_estimators': [50, 100],
    'rf__max_depth': [None, 10],
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 'auto'],
    'knn__n_neighbors': [3, 5, 7],
    'final_estimator__C': [0.1, 1, 10]
}

## Create Stacking Classifier and Grid Search

In [5]:
stacking = StackingClassifier(estimators=base_learners,
                              final_estimator=LogisticRegression(),
                              cv=5)

grid_search = GridSearchCV(stacking, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best parameters: {'final_estimator__C': 0.1, 'knn__n_neighbors': 3, 'rf__max_depth': None, 'rf__n_estimators': 50, 'svm__C': 10, 'svm__gamma': 'scale'}
Best CV AUC: 0.9942105263157895


## Evaluate on Test Set

In [6]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1]

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test Precision:", precision_score(y_test, y_pred))
print("Test Recall:", recall_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Test AUC:", roc_auc_score(y_test, y_proba))

Test Accuracy: 0.7213114754098361
Test Precision: 0.7857142857142857
Test Recall: 0.8979591836734694
Test F1: 0.8380952380952381
Test AUC: 0.6258503401360545


 Show confusion matrix

In [7]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(f"True Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")


Confusion Matrix:
True Negatives: 0, False Positives: 12
False Negatives: 5, True Positives: 44


## Save Final Model

In [8]:
joblib.dump(best_model, 'models/stacking_final.joblib')
print(" Final model saved to models/stacking_final.joblib")

 Final model saved to models/stacking_final.joblib


Save the best parameters

In [9]:
import json
best_params = grid_search.best_params_
with open('models/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=4)
print(" Best parameters saved to models/best_params.json")

 Best parameters saved to models/best_params.json


Quick Interpretation

In [10]:
print(" MODEL INTERPRETATION")
print("="*50)

# Extract individual models
rf_model = best_model.named_estimators_['rf']
svm_model = best_model.named_estimators_['svm']
knn_model = best_model.named_estimators_['knn']
meta_model = best_model.final_estimator_

print(f"\n Random Forest: {rf_model.n_estimators} trees, max_depth={rf_model.max_depth}")
print(f" SVM: C={svm_model.C}, gamma={svm_model.gamma}")
print(f" k-NN: {knn_model.n_neighbors} neighbors")
print(f" Meta-learner: Logistic Regression with C={meta_model.C}")

# Feature importance from Random Forest
feature_names = []
with open('data/processed/selected_features.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:10]

print("\n Top 10 Most Important Features:")
for i, idx in enumerate(indices):
    print(f"{i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

 MODEL INTERPRETATION

 Random Forest: 50 trees, max_depth=None
 SVM: C=10, gamma=scale
 k-NN: 3 neighbors
 Meta-learner: Logistic Regression with C=0.1

 Top 10 Most Important Features:
1. PPE: 0.1545
2. MDVP:Fo(Hz): 0.1401
3. RPDE: 0.1308
4. spread2: 0.1135
5. spread1: 0.1027
6. MDVP:Flo(Hz): 0.0430
7. MDVP:Fhi(Hz): 0.0424
8. MDVP:Jitter(Abs): 0.0412
9. NHR: 0.0330
10. MDVP:Jitter(%): 0.0287


## Done
Proceed to Week 8 (Model Evaluation & Interpretation).